# Data Collection

In [1]:
import requests
from bs4 import BeautifulSoup

In [3]:
page_count = 1
while True:
    res = requests.get(f"https://www.flipkart.com/search?q=laptops&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page={page_count}")
    soup = BeautifulSoup(res.text, 'html.parser')
    parent_div = soup.find('div', class_='QSCKDh eRsYMo')
    if parent_div:
        child_div = parent_div.find('div', class_='QSCKDh dLgFEE')
        main_div = child_div.find_all('div', class_='lvJbLV col-12-12')
        if not main_div:
            print("No valid pages anymore")
            break
        with open(f"laptop_data/Laptop_data_{page_count}", 'w', encoding='utf-8') as file:
            file.write(res.text)
            print(f"Downloaded data from page {page_count}")
    page_count += 1


KeyboardInterrupt: 

In [4]:
all_laptops = []

for i in range(1, 42):
    with open(f"laptop_data/Laptop_data_{i}", encoding='utf-8') as data:
        soup = BeautifulSoup(data, 'html.parser')
        main_div = soup.find('div', 'QSCKDh dLgFEE')
        cards = main_div.find_all('div', 'lvJbLV col-12-12')
        
        for card in cards:
            record = {
                'brand': None,
                'price': None,
                'processor': None,
                'ram_gb': None,
                'storage_gb': None,
                'storage_type': None,
                'screen_size': None,
                'os': None
            }
            
            title = card.find('div', 'RG5Slk')
            if title:
                title_tag = title.get_text().split()[0]
                record['brand'] = title_tag
            price = card.find('div', 'hZ3P6w DeU9vF')
            
            if price:
                price_tag = price.get_text().replace("₹", "").replace(",", "")
                record['price'] = price_tag
            
            info_cards = card.find_all('li', 'DTBslk')
            for card in info_cards:
                text = card.get_text().lower()
                if "ram" in text:
                    words = text.split()
                    for w in words:
                        digits = ''.join(c for c in w if c.isdigit())
                        if digits:
                            record['ram_gb'] = int(digits)
                            break
                    else:
                        record['ram_gb'] = None
        
                elif "ssd" in text or "hdd" in text:
                    record['storage_gb'] = int(text.split()[0])
                    record['storage_type'] = "SSD" if "ssd" in text else "HDD"
        
                elif "processor" in text or "core" in text or "ryzen" in text:
                    record['processor'] = text
                    
                elif "inch" in text:
                    record['screen_size'] = float(text.split()[0])
        
                elif "windows" in text or "dos" in text or "mac" in text:
                    record['os'] = text

            all_laptops.append(record)
                
print(all_laptops)           

[{'brand': 'ASUS', 'price': '37990', 'processor': 'intel core i3 processor (13th gen)', 'ram_gb': 8, 'storage_gb': 512, 'storage_type': 'SSD', 'screen_size': 35.56, 'os': 'windows 11 home operating system'}, {'brand': 'ASUS', 'price': '39990', 'processor': 'intel core i3 processor (13th gen)', 'ram_gb': 16, 'storage_gb': 512, 'storage_type': 'SSD', 'screen_size': 39.62, 'os': 'windows 11 home operating system'}, {'brand': 'Lenovo', 'price': '10999', 'processor': 'mediatek kompanio 520 processor', 'ram_gb': 4, 'storage_gb': None, 'storage_type': None, 'screen_size': 29.46, 'os': None}, {'brand': 'ASUS', 'price': '14999', 'processor': 'intel celeron dual core processor', 'ram_gb': 4, 'storage_gb': None, 'storage_type': None, 'screen_size': 35.56, 'os': None}, {'brand': 'HP', 'price': '69990', 'processor': 'intel core i7 processor (13th gen)', 'ram_gb': 16, 'storage_gb': 512, 'storage_type': 'SSD', 'screen_size': 39.62, 'os': 'windows 11 home operating system'}, {'brand': 'ASUS', 'price':

In [5]:
all_laptops = []

for i in range(1, 42):
    with open(f"laptop_data/Laptop_data_{i}", encoding='utf-8') as data:
        soup = BeautifulSoup(data, 'html.parser')
        main_div = soup.find('div', class_='QSCKDh dLgFEE')  
        if not main_div:
            continue
            
        cards = main_div.find_all('div', class_='lvJbLV col-12-12')
        
        for product_card in cards:                  
            record = {                              
                'brand': None,
                'price': None,
                'processor': None,
                'ram_gb': None,
                'storage_gb': None,
                'storage_type': None,
                'screen_size': None,
                'os': None
            }
            
            # Brand
            title = product_card.find('div', class_='RG5Slk')
            if title:
                record['brand'] = title.get_text().split()[0]
            
            # Price
            price_tag = product_card.find('div', class_='hZ3P6w DeU9vF')
            if price_tag:
                record['price'] = price_tag.get_text().replace("₹", "").replace(",", "").strip()
            
            # Specs
            info_items = product_card.find_all('li', class_='DTBslk')
            for item in info_items:          
                text = item.get_text().lower().strip()
                
                if "ram" in text:
                    words = text.split()
                    for w in words:
                        digits = ''.join(c for c in w if c.isdigit())
                        if digits:
                            record['ram_gb'] = int(digits)
                            break
                
                elif "ssd" in text or "hdd" in text:
                    words = text.split()
                    for w in words:
                        digits = ''.join(c for c in w if c.isdigit())
                        if digits:
                            record['storage_gb'] = int(digits)
                            break
                    record['storage_type'] = "SSD" if "ssd" in text else "HDD"
                
                elif any(kw in text for kw in ["processor", "core", "ryzen", "intel", "amd"]):
                    record['processor'] = text.strip()
                
                elif "inch" in text:
                    words = text.split()
                    for w in words:
                        try:
                            record['screen_size'] = float(w)
                            break
                        except ValueError:
                            pass
                
                elif any(kw in text for kw in ["windows", "dos", "mac", "linux", "chrome"]):
                    record['os'] = text.strip()
            
            if record['brand'] or record['price']:
                all_laptops.append(record)

print(len(all_laptops))
print(all_laptops[:5]) 

984
[{'brand': 'ASUS', 'price': '37990', 'processor': 'intel core i3 processor (13th gen)', 'ram_gb': 8, 'storage_gb': 512, 'storage_type': 'SSD', 'screen_size': 35.56, 'os': 'windows 11 home operating system'}, {'brand': 'ASUS', 'price': '39990', 'processor': 'intel core i3 processor (13th gen)', 'ram_gb': 16, 'storage_gb': 512, 'storage_type': 'SSD', 'screen_size': 39.62, 'os': 'windows 11 home operating system'}, {'brand': 'Lenovo', 'price': '10999', 'processor': 'mediatek kompanio 520 processor', 'ram_gb': 4, 'storage_gb': None, 'storage_type': None, 'screen_size': 29.46, 'os': 'chrome operating system'}, {'brand': 'ASUS', 'price': '14999', 'processor': 'intel celeron dual core processor', 'ram_gb': 4, 'storage_gb': None, 'storage_type': None, 'screen_size': 35.56, 'os': 'chrome operating system'}, {'brand': 'HP', 'price': '69990', 'processor': 'intel core i7 processor (13th gen)', 'ram_gb': 16, 'storage_gb': 512, 'storage_type': 'SSD', 'screen_size': 39.62, 'os': 'windows 11 home 

# Now that we have gathered the data - Let's perfrom data analysis on this data

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
data = pd.DataFrame(all_laptops)
data.head()

,brand,price,processor,ram_gb,storage_gb,storage_type,screen_size,os
0,ASUS,37990,intel core i3 processor (13th gen),8,512.0,SSD,35.56,windows 11 home operating system
1,ASUS,39990,intel core i3 processor (13th gen),16,512.0,SSD,39.62,windows 11 home operating system
2,Lenovo,10999,mediatek kompanio 520 processor,4,NaN,None,29.46,chrome operating system
3,ASUS,14999,intel celeron dual core processor,4,NaN,None,35.56,chrome operating system
4,HP,69990,intel core i7 processor (13th gen),16,512.0,SSD,39.62,windows 11 home operating system


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 984 entries, 0 to 983
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   brand         984 non-null    object 
 1   price         984 non-null    object 
 2   processor     984 non-null    object 
 3   ram_gb        984 non-null    int64  
 4   storage_gb    962 non-null    float64
 5   storage_type  962 non-null    object 
 6   screen_size   983 non-null    float64
 7   os            982 non-null    object 
dtypes: float64(2), int64(1), object(5)
memory usage: 61.6+ KB


In [9]:
data = data.dropna()

In [10]:
data = data[~data['os'].str.contains("built-in apps")]

In [11]:
data['price'] = data['price'].astype('int64')

In [12]:
def simplify_os(os_text):
    if "windows 11" in os_text.lower():
        return 'Windows 11'
    if "windows 10" in os_text.lower():
        return "Windows 10"
    if "windows" in os_text.lower():
        return "Windows"
    if "chrome" in os_text.lower():
        return "Chrome OS"
    return "Other"

data['os'] = data['os'].apply(simplify_os)

In [13]:
data['os'].value_counts()

os
Windows 11    925
Windows 10     12
Windows         7
Chrome OS       1
Name: count, dtype: int64

In [14]:
data['storage_gb'].value_counts()

storage_gb
512.0    778
1.0      131
256.0     29
2.0        4
128.0      3
Name: count, dtype: int64

In [15]:
def parse_storage(storage):
    if storage < 10:
        storage = storage * 1024
    return storage

data['storage_gb'] = data['storage_gb'].apply(parse_storage)

In [16]:
data['processor'] = data['processor'].str.lower()
def get_processor_brand(text):
    if 'intel' in text:
        return "intel"
    elif 'amd' in text:
        return 'amd'
    else:
        return 'other'

data['processor_brand'] = data['processor'].apply(get_processor_brand)

In [17]:
data['processor_brand'].value_counts()

processor_brand
intel    750
amd      172
other     23
Name: count, dtype: int64

In [18]:
import re

def get_processor_series(text):
    # Intel series
    intel_match = re.search(r"i[3579]", text)
    if intel_match:
        return intel_match.group()
    # AMD Ryzen series
    amd_match = re.search(r"ryzen\s*\d", text)
    if amd_match:
        return amd_match.group()
    if "core 5" in text:
        return 'i5'
    if "core ultra 5" in text:
        return 'i5'
    if "core 7" in text:
        return 'i7'
    if "core ultra 7" in text:
        return 'i7'
    if 'snapdragon' in text:
        return 'snapdragon'
    if "celeron" in text:
        return "budget"
    if "core 9" in text:
        return 'i9'
    if "core ultra 9" in text:
        return 'i9'
    if "core 3" in text:
        return 'i3'
    if "core ultra 3" in text:
        return 'i3'
    
    return "other"

data["processor_series"] = data["processor"].apply(get_processor_series)

In [19]:
data["processor_series"].value_counts()

processor_series
i5            354
i3            214
i7            140
ryzen 5        97
ryzen 3        40
ryzen 7        26
snapdragon     24
budget         18
i9             16
other          15
ryzen 9         1
Name: count, dtype: int64

In [20]:
def get_generation(text):
    match = re.search(r"(\d+)(st|nd|rd|th)\s*gen", text)
    if match:
        return int(match.group(1))
    return None

data["generation"] = data["processor"].apply(get_generation)

In [21]:
data['generation'].value_counts()

generation
13.0    354
12.0     97
14.0     59
11.0     24
10.0      5
7.0       2
1.0       1
8.0       1
5.0       1
3.0       1
Name: count, dtype: int64

In [22]:
data["generation"] = data["generation"].fillna(0)
data["generation"] = data["generation"].astype(int)

In [23]:
data = data.drop(columns='processor')

In [24]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 945 entries, 0 to 983
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   brand             945 non-null    object 
 1   price             945 non-null    int64  
 2   ram_gb            945 non-null    int64  
 3   storage_gb        945 non-null    float64
 4   storage_type      945 non-null    object 
 5   screen_size       945 non-null    float64
 6   os                945 non-null    object 
 7   processor_brand   945 non-null    object 
 8   processor_series  945 non-null    object 
 9   generation        945 non-null    int32  
dtypes: float64(2), int32(1), int64(2), object(5)
memory usage: 77.5+ KB


In [25]:
data['storage_type'].value_counts()

storage_type
SSD    942
HDD      3
Name: count, dtype: int64

In [26]:
data['ram_gb'].value_counts()

ram_gb
16    516
8     269
32     93
24     53
12      8
4       6
Name: count, dtype: int64

In [29]:
data.to_csv("laptop_data2.csv", index=False)

# Now We can train the model using this data

In [314]:
X = data.drop(columns='price')
y = data['price']

In [316]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.3)

In [318]:
data.columns

Index(['brand', 'price', 'ram_gb', 'storage_gb', 'storage_type', 'screen_size',
       'os', 'processor_brand', 'processor_series', 'generation'],
      dtype='object')

In [320]:
cat_cols = ['brand', 'storage_type', 'os', 'processor_brand', 'processor_series']
nums_cols = ['price', 'ram_gb', 'storage_gb', 'screen_size']